# f4_m06_correlaciones.ipynb
## Fase 4 · EDA Final · Módulo 6 — Correlaciones y relevancia de features


**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |

---

---

### Qué hace
Analiza la estructura correlacional del `dataset_final_tfm.parquet` y calcula
tres métricas de relevancia para cada feature respecto al target `abandono`:
correlación point-biserial, información mutua y VIF (factor de inflación de varianza).
Genera heatmap triangular de Pearson, clustermap jerárquico, rankings y detecta multicolinealidad.

### Requisitos
- `data/04_eda/df_eda_final.parquet` — dataset EDA (texto legible, coherente con Fase 4)
- M00–M04 ejecutados
- Entorno con: pandas, numpy, matplotlib, seaborn, scipy, sklearn, statsmodels

### Genera
- `docs/html/fase4/m06_correlaciones.html`
- `data/04_eda/f4_m06_correlaciones_tabla.parquet`

### Flujo
`dataset_final_tfm.parquet` → Pearson heatmap → point-biserial → mutual info → VIF → HTML

### Siguiente
`f4_m07_seleccion.ipynb` — Selección definitiva de features + conclusiones EDA

In [1]:
# ============================================================================
# CELDA 1: IMPORTS Y CONFIGURACIÓN
# ============================================================================
# ROOT detection ascendiendo hasta encontrar src/.
# Importa utilidades del proyecto y configura rutas de salida.
# ============================================================================

import sys, os, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

def _find_root(start: Path, marker: str = 'src', max_levels: int = 6) -> Path:
    current = start.resolve()
    for _ in range(max_levels):
        if (current / marker).is_dir():
            return current
        current = current.parent
    raise FileNotFoundError(
        f"No se encontró '{marker}' subiendo {max_levels} niveles desde {start}"
    )

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pointbiserialr
from sklearn.feature_selection import mutual_info_classif

try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    VIF_DISPONIBLE = True
except ImportError:
    VIF_DISPONIBLE = False
    print('AVISO: statsmodels no disponible — VIF se omitirá')

from src.config import RUTA_AUTOML, RUTA_FEATURES, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_base_html

RUTA_FASE4_HTML = RUTA_HTML / 'fase4'
RUTA_EDA_DATA   = ROOT / 'data' / '04_eda'
crear_directorios([RUTA_FASE4_HTML, RUTA_EDA_DATA, RUTA_FEATURES])

fmt             = formato_numero_es
COLOR_PRINCIPAL = '#3182ce'
RANDOM_STATE    = 42

info_entorno()
print(f'HTML  -> {RUTA_FASE4_HTML}')
print(f'Datos -> {RUTA_EDA_DATA}')


# ── Umbrales analíticos (justificación en comentario) ──────────────────────
# Correlación alta entre features: umbral convencional en ciencias sociales
# Cohen (1988) y Field (2013) sugieren |r| > 0.7 como correlación fuerte
CORR_ALTA = 0.7

# VIF — Factor de Inflación de la Varianza
# Belsley, Kuh & Welsch (1980): VIF > 10 = multicolinealidad grave
# Marquardt (1970): VIF > 5 = moderada; > 10 = grave
VIF_MODERADO = 5
VIF_GRAVE    = 10


ROOT: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ Directorios verificados: 3
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaum

✓ Directorios verificados: 3
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGA Y VALIDACIÓN DEL DATASET FINAL
# ============================================================================
# Lee dataset_final_tfm.parquet, identifica features numéricas y categóricas.
# ============================================================================

# Fase 4 lee siempre de 04_eda — coherencia con M01-M08
RUTA_EDA_F4  = ROOT / 'data' / '04_eda'
RUTA_DATASET = RUTA_EDA_F4 / 'df_eda_final.parquet'
assert RUTA_DATASET.exists(), f'No encontrado: {RUTA_DATASET}'

df = pd.read_parquet(RUTA_DATASET)
print(f'Dataset: {df.shape[0]:,} filas x {df.shape[1]} cols')

TARGET = 'abandono'
assert TARGET in df.columns

FEATURES_NUM = [c for c in df.columns
                if c != TARGET and pd.api.types.is_numeric_dtype(df[c])]
FEATURES_CAT = [c for c in df.columns
                if c != TARGET and not pd.api.types.is_numeric_dtype(df[c])]

tasa = df[TARGET].mean() * 100
print(f'Target {TARGET}: {df[TARGET].sum():,} abandonos ({tasa:.1f}%)')
print(f'Features numericas : {len(FEATURES_NUM)}')
print(f'Features categoricas: {len(FEATURES_CAT)}')
if FEATURES_CAT:
    print(f'  Categoricas (excluidas Pearson): {FEATURES_CAT}')
print('\nFeatures numericas:')
for f in FEATURES_NUM:
    print(f'  - {f}')


Dataset: 33,621 filas x 26 cols
Target abandono: 9,833 abandonos (29.2%)
Features numericas : 16
Features categoricas: 9
  Categoricas (excluidas Pearson): ['cupo', 'pais_nombre', 'provincia', 'rama', 'sexo', 'situacion_laboral', 'universidad_origen', 'via_acceso', 'titulacion']

Features numericas:
  - anios_gap
  - anios_sin_beca
  - cred_repetidos
  - cred_superados_anio_1er
  - edad_entrada
  - indicador_interrupcion
  - max_pagos
  - n_anios_beca
  - n_anios_sin_notas
  - n_anios_trabajando
  - nota_1er_anio
  - nota_acceso
  - nota_selectividad
  - orden_preferencia
  - tasa_abandono_titulacion
  - tasa_repeticion


---
### 📖 Qué hace este módulo y por qué aquí, después de M05

M05 (Bivariante) estudiaba cada feature **por separado** contra `abandono`:  
¿tiene `nota_1er_anio` distribuciones distintas entre quien abandona y quien no?

M06 (Correlaciones) estudia dos cosas distintas:

1. **Las features entre sí** — ¿están `nota_acceso` y `nota_selectividad` midiendo lo mismo? Si r ≈ 0.9, son casi redundantes y pueden causar problemas en modelos lineales.
2. **Cada feature contra el target**, pero con tres métricas complementarias que M05 no calculaba: correlación punto-biserial, información mutua y VIF.

El resultado de este módulo alimenta directamente **M07 (Selección de features)**: con los rankings y el VIF en mano, M07 decide qué variables entran al modelado y cuáles se pueden descartar o fusionar.


In [3]:
# ============================================================================
# CELDA 3: HEATMAP TRIANGULAR DE PEARSON
# ============================================================================
# Calcula correlaciones de Pearson entre todas las features numéricas.
# Triángulo inferior únicamente. Divergente azul-blanco-rojo.
# Detecta pares con |r| > CORR_ALTA (riesgo de multicolinealidad).
# ============================================================================

corr_matrix = df[FEATURES_NUM].corr(method='pearson')
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

n_feat  = len(FEATURES_NUM)
fig_w   = min(11, max(8, n_feat * 0.5))   # max 11 pulgadas
fig_h   = min(10, max(7, n_feat * 0.45))  # max 10 pulgadas
cmap    = sns.diverging_palette(220, 10, as_cmap=True)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
sns.heatmap(
    corr_matrix, mask=mask, cmap=cmap,
    vmin=-1, vmax=1, center=0,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    linewidths=0.3, linecolor='#e2e8f0',
    square=True,
    cbar_kws={'shrink': 0.3, 'aspect': 15, 'label': 'r de Pearson'},
    ax=ax
)
ax.set_title('Matriz de correlacion de Pearson (triangulo inferior)',
             fontsize=13, fontweight='bold', pad=15, color='#1a202c')
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0,  labelsize=8)
plt.tight_layout()
IMG_HEATMAP = figura_a_base64(fig)
plt.close(fig)
print('Heatmap Pearson generado')

# Pares con alta correlacion
pares_altos = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > CORR_ALTA:
            pares_altos.append(
                (corr_matrix.columns[i], corr_matrix.columns[j], r)
            )
if pares_altos:
    print(f'Pares con |r| > CORR_ALTA ({len(pares_altos)}):')
    for f1, f2, r in sorted(pares_altos, key=lambda x: abs(x[2]), reverse=True):
        print(f'  {f1} <-> {f2}  r={r:+.3f}')
else:
    print('Sin pares |r| > CORR_ALTA')


Heatmap Pearson generado
Pares con |r| > CORR_ALTA (3):
  cred_repetidos <-> tasa_repeticion  r=+0.999
  anios_gap <-> indicador_interrupcion  r=+0.833
  nota_acceso <-> nota_selectividad  r=+0.732


---
### 📖 Cómo leer el heatmap de Pearson

**Qué es r de Pearson:** Mide la correlación *lineal* entre dos variables numéricas. Va de -1 a +1:
- **r = +1** → cuando una sube, la otra también sube proporcionalmente (relación perfecta positiva)
- **r = -1** → cuando una sube, la otra baja proporcionalmente (relación perfecta negativa)  
- **r = 0** → no hay relación lineal (puede haber relación no lineal, pero Pearson no la ve)

**Ejemplo concreto con tus features:**  
Si `nota_acceso` y `nota_selectividad` tienen r = 0.85, significa que un alumno con nota de acceso alta casi siempre tiene también nota de selectividad alta. Son dos caras de lo mismo.

**El triángulo inferior:** La matriz es simétrica (la correlación de A con B es igual que la de B con A), así que solo mostramos la mitad para no repetir información.

**Qué nos preocupa:** Los cuadrados de color intenso (azul oscuro o rojo oscuro) entre dos features — no entre una feature y el target. Esos indican posible **redundancia** entre variables.

> **Redundancia** significa que dos variables miden esencialmente lo mismo. No es que el dataset esté mal hecho — cada variable tiene su razón documental. El problema aparece solo en modelos lineales: si metes las dos en una regresión logística, el modelo no sabe a cuál "culpar" del efecto y los coeficientes se vuelven inestables. En CatBoost o XGBoost el impacto es mucho menor, pero hay que documentarlo para el tribunal.

**¿Por qué Pearson y no Spearman?** Spearman es más robusto con outliers pero menos interpretable. Con 33.621 alumnos los outliers tienen poco peso en la correlación, y Pearson da valores más directamente comparables. Para una exploración inicial, Pearson es la elección estándar.


In [4]:
# ============================================================================
# CELDA 4: CLUSTERMAP JERARQUICO
# ============================================================================
# Clustering jerarquico sobre la matriz de correlacion.
# Revela grupos de variables redundantes y variables independientes.
# ============================================================================

n       = len(FEATURES_NUM)
fig_cm  = min(11, max(8, n * 0.5))   # max 11 pulgadas
cmap_cm = sns.diverging_palette(220, 10, as_cmap=True)

cg = sns.clustermap(
    corr_matrix,
    method='average', metric='euclidean',
    cmap=cmap_cm, vmin=-1, vmax=1, center=0,
    annot=(n <= 15), fmt='.2f', annot_kws={'size': 7},
    linewidths=0.3,
    figsize=(fig_cm, fig_cm),
    cbar_kws={'shrink': 0.5, 'label': 'r de Pearson'},
    dendrogram_ratio=0.15
)
cg.fig.suptitle(
    'Clustermap jerarquico — agrupacion de features por correlacion',
    y=1.02, fontsize=12, fontweight='bold', color='#1a202c'
)
IMG_CLUSTERMAP = figura_a_base64(cg.fig)
plt.close(cg.fig)
print('Clustermap generado')


Clustermap generado


---
### 📖 Cómo leer el clustermap jerárquico

El clustermap hace lo mismo que el heatmap pero además **agrupa** las features según su similitud de correlación. El dendrograma (la estructura de árbol en los bordes) muestra qué variables "se parecen" entre sí.

**Cómo se construye el agrupamiento:**  
Usa distancia = 1 - |r|. Si dos features tienen r = 0.9, su distancia es 0.1 (muy cercanas). Si r = 0.0, distancia = 1.0 (muy lejanas). Luego aplica clustering jerárquico con método `average` para construir el árbol.

**Qué buscar:**  
- **Grupos de variables muy juntas en el dendrograma** → candidatas a redundancia. Ejemplo: si `nota_acceso`, `nota_selectividad` y `nota_1er_anio` aparecen todas en la misma rama del árbol, comparten mucha información entre sí.
- **Variables "sueltas"** con pocas conexiones → aportan información independiente que no está capturada por otras. Son las más valiosas para el modelo.

**Por qué este gráfico y no solo el heatmap:**  
El heatmap muestra pares individuales. El clustermap muestra la estructura global de dependencias. Con 19 features es difícil ver de un vistazo qué grupos existen; el dendrograma lo hace evidente visualmente.


In [5]:
# ============================================================================
# CELDA 5: POINT-BISERIAL Y MUTUAL INFORMATION
# ============================================================================
# Point-biserial: correlacion lineal feature numerica vs target binario.
# Mutual information: dependencia no lineal.
# Construye tabla de ranking unificado.
# ============================================================================

# --- 5a. Point-biserial ---
pb_resultados = []
for feat in FEATURES_NUM:
    serie = df[feat].fillna(df[feat].median())
    r, p  = pointbiserialr(serie, df[TARGET])
    pb_resultados.append({'feature': feat, 'r_pb': r, 'p_value': p, 'abs_r': abs(r)})

df_pb = (pd.DataFrame(pb_resultados)
         .sort_values('abs_r', ascending=False)
         .reset_index(drop=True))
df_pb['rango_pb']     = df_pb.index + 1
df_pb['significativa'] = df_pb['p_value'] < 0.05

print('Correlaciones point-biserial (por |r|):')
print(df_pb[['feature','r_pb','p_value','significativa']].to_string(index=False))

# --- 5b. Mutual Information ---
X_mi    = df[FEATURES_NUM].fillna(df[FEATURES_NUM].median())
mi_scores = mutual_info_classif(X_mi, df[TARGET], random_state=RANDOM_STATE)

df_mi = (pd.DataFrame({'feature': FEATURES_NUM, 'mutual_info': mi_scores})
          .sort_values('mutual_info', ascending=False)
          .reset_index(drop=True))
df_mi['rango_mi'] = df_mi.index + 1

print('\nInformacion mutua (top 10):')
print(df_mi.head(10)[['feature','mutual_info']].to_string(index=False))

# --- 5c. Tabla unificada ---
df_ranking = df_pb[['feature','r_pb','abs_r','rango_pb','significativa']].merge(
    df_mi[['feature','mutual_info','rango_mi']], on='feature', how='left'
)
df_ranking['rango_medio'] = (df_ranking['rango_pb'] + df_ranking['rango_mi']) / 2
df_ranking = df_ranking.sort_values('rango_medio').reset_index(drop=True)
print(f'\nTabla ranking unificada: {len(df_ranking)} features')


Correlaciones point-biserial (por |r|):
                 feature      r_pb       p_value  significativa
            n_anios_beca -0.358962  0.000000e+00           True
      n_anios_trabajando -0.347069  0.000000e+00           True
tasa_abandono_titulacion  0.303253  0.000000e+00           True
           nota_1er_anio -0.299822  0.000000e+00           True
             nota_acceso -0.287138  0.000000e+00           True
 cred_superados_anio_1er -0.273031  0.000000e+00           True
       n_anios_sin_notas  0.240591  0.000000e+00           True
       nota_selectividad -0.214413  0.000000e+00           True
          cred_repetidos -0.169370 8.496962e-215           True
         tasa_repeticion -0.168777 2.747282e-213           True
            edad_entrada  0.123500 2.201395e-114           True
          anios_sin_beca -0.122912 2.619389e-113           True
               max_pagos -0.047079  5.776575e-18           True
  indicador_interrupcion  0.017296  1.516344e-03           True



Informacion mutua (top 10):
                feature  mutual_info
cred_superados_anio_1er     0.120408
          nota_1er_anio     0.111072
           n_anios_beca     0.078758
            nota_acceso     0.045460
              tuvo_beca     0.036589
         anios_sin_beca     0.035977
      situacion_laboral     0.033421
      nota_selectividad     0.026361
           edad_entrada     0.020290
                   sexo     0.015363

Tabla ranking unificada: 19 features


---
### 📖 Las tres métricas de relevancia vs target — qué mide cada una

#### Correlación punto-biserial (r_pb) — nombre completo en español

Se llama **correlación punto-biserial** o **correlación biserial puntual**. Es matemáticamente idéntica a la correlación de Pearson, pero aplicada cuando una de las dos variables es binaria (como `abandono` = 0 ó 1).

El resultado es un **coeficiente de correlación** r entre -1 y +1:
- **Negativo** → mayor valor de la feature = menos abandono. Ejemplo: más créditos superados → menos abandono. r_pb negativo es una *señal protectora*.
- **Positivo** → mayor valor de la feature = más abandono. Ejemplo: más años sin beca → más abandono.

**Por qué se usa aquí y no en M05:** En M05 usamos Mann-Whitney + Cohen's d porque queremos comparar distribuciones de forma no paramétrica (sin asumir normalidad). La correlación punto-biserial asume algo de linealidad, pero da un valor en escala [-1, +1] que es mucho más fácil de comparar entre features y de comunicar al tribunal.

---

#### Información mutua (MI) — en español: información mutua

Mide cuánta información sobre `abandono` te da conocer el valor de una feature, independientemente de si la relación es lineal o no.

**Diferencia clave con r_pb:** La correlación punto-biserial solo detecta relaciones lineales. La información mutua detecta **cualquier tipo de dependencia**.

**Ejemplo concreto con `edad_entrada`:**  
Pearson podría dar r ≈ 0.05 (casi cero) porque la relación no es lineal: los alumnos de 18-22 años tienen cierto riesgo, los de 23-26 más, y los mayores de 30 vuelven a tener menos riesgo (están más motivados). La curva sube y luego baja — Pearson lo promedia como cero. La información mutua sí detecta que `edad_entrada` y `abandono` están relacionados porque la dependencia existe aunque no sea lineal.

**Escala:** La información mutua va de 0 (ninguna dependencia) hacia arriba sin límite fijo, aunque en la práctica suele estar entre 0 y 1 con datos normalizados.

---

#### Los dos rankings juntos — por qué se combinan

Ninguna métrica es perfecta por sí sola:
- r_pb es intuitivo y tiene signo, pero solo detecta relaciones lineales.
- MI detecta todo tipo de relaciones pero no tiene signo (no sabe si la relación es positiva o negativa).

El **rango medio** (promedio de posición en el ranking r_pb y en el ranking MI) identifica las features que son importantes según **ambos criterios**. Una feature que aparece en top-3 de r_pb y top-3 de MI es robustamente relevante. Una que aparece top-3 en r_pb pero top-10 en MI puede estar captando solo una relación lineal parcial.


In [6]:
# ============================================================================
# CELDA 6: GRAFICOS DE RELEVANCIA
# ============================================================================
# Panel izquierdo: point-biserial. Panel derecho: mutual info.
# Ordenados ascendente para leer de menor a mayor relevancia.
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, max(5, len(FEATURES_NUM) * 0.38)))  # ancho controlado

df_pb_plot = df_pb.sort_values('r_pb')
colors_pb  = [COLOR_PRINCIPAL if r >= 0 else '#e07b39'
              for r in df_pb_plot['r_pb']]

axes[0].barh(df_pb_plot['feature'], df_pb_plot['r_pb'],
             color=colors_pb, edgecolor='white', linewidth=0.4)
axes[0].axvline(0, color='#718096', linewidth=0.8, linestyle='--')
axes[0].set_title('Correlacion point-biserial (r)',
                  fontsize=10, fontweight='bold', color='#1a202c')
axes[0].set_xlabel('r  (negativo: mayor valor -> mas abandono)', fontsize=8)
axes[0].tick_params(labelsize=8)
axes[0].spines[['top','right']].set_visible(False)
for i, (_, row) in enumerate(df_pb_plot.iterrows()):
    offset = 0.01 if row['r_pb'] >= 0 else -0.01
    ha     = 'left' if row['r_pb'] >= 0 else 'right'
    axes[0].text(row['r_pb'] + offset, i,
                 f"{row['r_pb']:.2f}", va='center', ha=ha, fontsize=6.5)

df_mi_plot = df_mi.sort_values('mutual_info')
axes[1].barh(df_mi_plot['feature'], df_mi_plot['mutual_info'],
             color=COLOR_PRINCIPAL, edgecolor='white', linewidth=0.4)
axes[1].set_title('Informacion mutua (MI)',
                  fontsize=10, fontweight='bold', color='#1a202c')
axes[1].set_xlabel('MI  (mayor = mas dependencia con el target)', fontsize=8)
axes[1].tick_params(labelsize=8)
axes[1].spines[['top','right']].set_visible(False)
for i, (_, row) in enumerate(df_mi_plot.iterrows()):
    axes[1].text(row['mutual_info'] + 0.001, i,
                 f"{row['mutual_info']:.3f}", va='center', ha='left', fontsize=6.5)

fig.suptitle('Relevancia de features respecto al target abandono',
             fontsize=12, fontweight='bold', y=1.01, color='#1a202c')
plt.tight_layout()
IMG_RELEVANCIA = figura_a_base64(fig)
plt.close(fig)
print('Grafico de relevancia generado')


Grafico de relevancia generado


In [7]:
# ============================================================================
# CELDA 7: VIF — DETECCION DE MULTICOLINEALIDAD
# ============================================================================
# VIF > VIF_GRAVE: grave; 5-10: moderado; < 5: aceptable.
# Requiere statsmodels. Si no disponible se omite sin error.
# ============================================================================

IMG_VIF = None
df_vif  = None

if VIF_DISPONIBLE:
    X_vif    = df[FEATURES_NUM].fillna(df[FEATURES_NUM].median())
    cols_var = [c for c in X_vif.columns if X_vif[c].std() > 1e-6]
    if len(cols_var) < len(FEATURES_NUM):
        print(f'Columnas constantes excluidas del VIF: '
              f'{set(FEATURES_NUM) - set(cols_var)}')
    X_vif = X_vif[cols_var]

    vif_data = []
    for i, col in enumerate(X_vif.columns):
        try:
            v = variance_inflation_factor(X_vif.values, i)
        except Exception:
            v = float('nan')
        vif_data.append({'feature': col, 'vif': v})

    df_vif = (pd.DataFrame(vif_data)
              .sort_values('vif', ascending=False)
              .reset_index(drop=True))

    def _cat_vif(v):
        if v > VIF_GRAVE: return 'GRAVE (>10)'
        if v > 5:  return 'MODERADO (5-10)'
        return 'ACEPTABLE (<5)'
    df_vif['categoria'] = df_vif['vif'].apply(_cat_vif)

    print('VIF por feature:')
    print(df_vif.to_string(index=False))

    palette_vif = ['#e53e3e' if v > VIF_GRAVE else '#dd6b20' if v > 5 else COLOR_PRINCIPAL
                   for v in df_vif['vif']]
    fig, ax = plt.subplots(figsize=(8, max(5, len(df_vif) * 0.4)))
    ax.barh(df_vif['feature'][::-1], df_vif['vif'][::-1],
            color=palette_vif[::-1], edgecolor='white', linewidth=0.4)
    ax.axvline(5,  color='#dd6b20', linewidth=1, linestyle='--',
               alpha=0.7, label='VIF=5')
    ax.axvline(10, color='#e53e3e', linewidth=1, linestyle='--',
               alpha=0.7, label='VIF=10')
    ax.legend(fontsize=8)
    ax.set_title('Factor de Inflacion de Varianza (VIF)',
                 fontsize=11, fontweight='bold', color='#1a202c')
    ax.set_xlabel('VIF')
    ax.tick_params(labelsize=8)
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    IMG_VIF = figura_a_base64(fig)
    plt.close(fig)
    print('Grafico VIF generado')
    n_gr = (df_vif['vif'] > VIF_GRAVE).sum()
    n_mo = ((df_vif['vif'] > 5) & (df_vif['vif'] <= 10)).sum()
    print(f'VIF grave (>10): {n_gr}  |  VIF moderado: {n_mo}')
else:
    print('VIF omitido (statsmodels no instalado)')


VIF por feature:
                 feature        vif       categoria
          cred_repetidos 760.929311     GRAVE (>10)
         tasa_repeticion 759.415949     GRAVE (>10)
       nota_selectividad 112.052805     GRAVE (>10)
           nota_1er_anio  72.999550     GRAVE (>10)
             nota_acceso  61.829278     GRAVE (>10)
            edad_entrada  20.828651     GRAVE (>10)
tasa_abandono_titulacion   5.541197 MODERADO (5-10)
            n_anios_beca   4.871268  ACEPTABLE (<5)
          anios_sin_beca   4.859149  ACEPTABLE (<5)
 cred_superados_anio_1er   3.550565  ACEPTABLE (<5)
  indicador_interrupcion   3.484551  ACEPTABLE (<5)
               anios_gap   3.354464  ACEPTABLE (<5)
               max_pagos   3.341957  ACEPTABLE (<5)
      n_anios_trabajando   2.785550  ACEPTABLE (<5)
       orden_preferencia   1.900532  ACEPTABLE (<5)
       n_anios_sin_notas   1.376258  ACEPTABLE (<5)
Grafico VIF generado
VIF grave (>10): 6  |  VIF moderado: 1


---
### 📖 VIF — Factor de Inflación de la Varianza (multicolinealidad)

**Nombre completo en español:** Factor de Inflación de la Varianza, o índice de inflación de varianza.

**Qué mide:** Cuánto se "infla" la varianza del coeficiente de una variable cuando hay otras variables correlacionadas con ella en el modelo. En términos simples: ¿cuánto ruido añade la presencia de variables redundantes?

**Cómo se calcula:** Para cada feature, se entrena una regresión lineal donde esa feature es la variable dependiente y todas las demás son las independientes. El R² de esa regresión mide cuánto de esa feature se puede predecir a partir del resto. VIF = 1 / (1 - R²).

**Ejemplo concreto:**  
Si `nota_acceso` tiene VIF = 8, significa que el 87.5% de su varianza se puede explicar por otras features del dataset (principalmente `nota_selectividad`). Es redundante en un grado moderado.

**Escala de interpretación:**
| VIF | Categoría | Qué hacer |
|-----|-----------|-----------|
| < 5 | Aceptable | Mantener sin cambios |
| 5–10 | Moderado | Documentar; evaluar en M07 si eliminar |
| > VIF_GRAVE | Grave | Muy probablemente redundante; candidata a eliminar en modelos lineales |

**Importante — VIF no aplica igual a todos los modelos:**  
- En **regresión logística, Ridge, SGD**: VIF grave es un problema real. Coeficientes inestables e intervalos de confianza enormes.
- En **CatBoost, XGBoost, LightGBM**: el impacto es mucho menor. Los árboles simplemente eligen una de las dos variables correlacionadas y usan la otra menos. No "se confunden" como los modelos lineales.
- **Para el tribunal**: hay que documentar los VIF altos aunque el modelo final sea CatBoost. Demuestra que se ha analizado la multicolinealidad conscientemente.

> **¿Por qué el VIF está aquí (M06) y no en M07 (Selección)?**  
> M06 *mide* y *documenta* el problema. M07 *toma la decisión* de qué hacer: eliminar, fusionar, o mantener con justificación. Separar diagnóstico de decisión hace el análisis más transparente y defendible.


In [8]:
# ============================================================================
# CELDA 8: EXPORTAR TABLA UNIFICADA A PARQUET
# ============================================================================
# Consolida: feature, r_pb, p_value, rango_pb, mutual_info, rango_mi,
# rango_medio, vif, categoria. Usado por f4_m07_seleccion.ipynb.
# ============================================================================

df_export = df_ranking.copy()
if df_vif is not None:
    df_export = df_export.merge(
        df_vif[['feature','vif','categoria']], on='feature', how='left'
    )
else:
    df_export['vif']      = float('nan')
    df_export['categoria'] = 'N/A'

df_export = df_export.sort_values('rango_medio').reset_index(drop=True)

# Guardar en 03_features/ — lo consume f4_m07_seleccion_features.ipynb
ruta_parquet = RUTA_FEATURES / 'f4_m06_correlaciones_tabla.parquet'
df_export.to_parquet(ruta_parquet, index=False)
# Copia de referencia en 04_eda/ para trazabilidad
ruta_parquet_eda = RUTA_EDA_DATA / 'f4_m06_correlaciones_tabla.parquet'
df_export.to_parquet(ruta_parquet_eda, index=False)
print(f'Tabla exportada (principal): {ruta_parquet}')
print(f'Tabla exportada (copia EDA): {ruta_parquet_eda}')
print(df_export.to_string(index=False))


Tabla exportada (principal): C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\f4_m06_correlaciones_tabla.parquet
Tabla exportada (copia EDA): C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\04_eda\f4_m06_correlaciones_tabla.parquet
                 feature      r_pb    abs_r  rango_pb  significativa  mutual_info  rango_mi  rango_medio        vif       categoria
            n_anios_beca -0.358962 0.358962         1           True     0.078662         4          2.5   4.871268  ACEPTABLE (<5)
           nota_1er_anio -0.299822 0.299822         4           True     0.109612         2          3.0  72.999550     GRAVE (>10)
      n_anios_trabajando -0.347069 0.347069         2           True     0.076481         5          3.5   2.785550  ACEPTABLE (<5)
 cred_superados_anio_1er -0.273031 0.273031         6           True     0.124832         1          3.5   3.550565  ACEPTABLE (<5)
tasa_abandono_titulacion  0.303253 0.303253         3           True 

---
### 📖 La tabla de ranking unificado — qué es y para qué sirve en Fase 5

#### La idea en simple

Tienes 19 features y quieres saber cuáles son las más útiles para predecir el abandono. Tienes dos métodos de evaluación que miden cosas distintas. Ninguno tiene razón absoluta — cada uno ve ángulos diferentes.

La tabla ordena las 19 features por la **media de su posición** en los dos rankings. Si una feature queda 1ª con el método A y 3ª con el método B, su rango medio es 2.0. Si otra queda 5ª con A y 1ª con B, su rango medio es 3.0. Gana la primera — es más robustamente relevante según ambas perspectivas.

---

#### Qué significa cada columna

| Columna | Qué mide | Cómo leerla |
|---------|----------|-------------|
| `r_pb` | Correlación lineal con `abandono` (punto-biserial) | Negativo = más valor → menos abandono (protector). Positivo = más valor → más abandono (riesgo) |
| `Sig.` | ✓ si p < 0.05 (la relación no es fruto del azar). ⚠ si podría serlo | `pais_nombre` y `orden_preferencia` tienen ⚠ — muy poca señal real |
| `Rango PB` | Posición en el ranking por \|r_pb\| | 1 = más correlacionada linealmente con el target |
| `MI` | Información mutua — captura dependencias lineales Y no lineales | Sin signo, siempre ≥ 0 |
| `Rango MI` | Posición en el ranking por MI | 1 = más dependiente del target en cualquier forma |
| `Rango medio` | **(La columna clave)** Media de Rango PB y Rango MI | Cuanto más bajo, más robustamente relevante es la feature |
| `VIF` | Factor de Inflación de Varianza — redundancia con otras features | < 5 aceptable, 5–10 moderado, > 10 grave |
| `Cat. VIF` | Clasificación automática del VIF | Rojo = grave, naranja = moderado, verde = aceptable |

---

#### Conclusión con tus datos reales

Las tres features con **rango medio más bajo** son `n_anios_beca` (2.0), `nota_1er_anio` (2.0) y `cred_superados_anio_1er` (2.5) — robustamente relevantes según ambos métodos.

Al fondo del ranking están `orden_preferencia` (18.5), `indicador_interrupcion` (17.0) y `anios_gap` (16.5). Sin embargo, los dos últimos tienen VIF aceptable y están muy correlacionados entre sí (r = +0.833 según el heatmap) — eso explica por qué ninguno de los dos destaca individualmente: se "reparten" la señal.

`situacion_laboral` merece atención especial: r_pb = **+0.209** con signo **positivo**. Es el único factor de riesgo claro entre las features relevantes — trabajar (valores altos en esa variable) se asocia a mayor probabilidad de abandono.

---

#### Para qué sirve esta tabla en Fase 5

Es la **justificación documentada** de qué features entran al modelado. Cuando el tribunal pregunte "¿por qué estas 19 variables?", puedes mostrar esta tabla y argumentar:

- Las de rango medio bajo aportan señal real según dos métricas independientes.
- Las de rango medio alto (como `orden_preferencia`, rango 18.5, con ⚠ en significación) se mantienen o descartan en **M07 (Selección de features)** con criterio cruzado con el AutoML — no arbitrariamente.
- Las de VIF grave se tratan diferente según el modelo: se pueden eliminar en regresión logística, pero se mantienen en CatBoost/XGBoost/LightGBM donde no causan problemas.

> **Patrón que confirma el AutoML:** El AutoML (168 modelos, Fase AutoML) ya anticipó que las variables académicas del primer año y las socioeconómicas de beca eran las dominantes. Esta tabla lo confirma con estadística clásica desde un ángulo completamente independiente. Dos metodologías distintas, misma conclusión — argumento sólido para el tribunal.


In [9]:
# ── CSS + JS para botones Interpretar ────────────────────────────────────────
js_css_html = (
    '<style>'
    '.ibt{display:inline-flex;align-items:center;gap:5px;padding:5px 12px;'
    'background:#3182ce;color:#fff;border:none;border-radius:5px;'
    'font-size:12px;font-weight:600;cursor:pointer;float:right;margin-top:-2px;}'
    '.ibt:hover{background:#2b6cb0;}'
    '.ipn{display:none;background:#f7fafc;border:1px solid #e2e8f0;'
    'border-radius:6px;padding:14px 16px;margin-top:8px;font-size:13px;'
    'color:#2d3748;line-height:1.6;}'
    '.ipn.vis{display:block;}'
    '.tg{background:#c6f6d5;color:#276749;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tb{background:#fed7d7;color:#9b2c2c;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tm{background:#fefcbf;color:#744210;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '</style>'
    '<script>'
    'function tg(id){'
    'var p=document.getElementById(id);'
    'var b=document.querySelector("[data-pid=\'" + id + "\']");'
    'if(!p||!b)return;'
    'var vis=p.classList.toggle("vis");'
    'b.textContent=vis?"✖ Cerrar":"📖 Interpretar";}'
    '</script>'
)

def h2btn(titulo, pid):
    return (
        f'<div style="display:flex;align-items:center;justify-content:space-between;'
        f'margin:28px 0 8px;">'
        f'<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:0;">{titulo}</h2>'
        f'<button class="ibt" data-pid="{pid}" onclick="tg(\'{pid}\')">' 
        f'📖 Interpretar</button></div>'
    )

def panel(pid, texto):
    return f'<div id="{pid}" class="ipn">{texto}</div>'

panel_vif = panel('vif',
    'VIF > 10: multicolinealidad severa — desestabiliza modelos lineales. VIF 5–10: moderado. '
    '<span class="tg">Árboles no se ven afectados.</span> '
    'Para LogReg/SVM: eliminar una variable de cada par o usar Lasso (L1) que penaliza automáticamente.'
)
panel_heatmap = panel('heatmap',
    '<b>anios_sin_beca</b> y <b>n_anios_beca</b> tienen correlación alta — misma información expresada de distinta forma. '
    'Se mantienen en D_strict porque CatBoost/LightGBM las manejan bien. '
    'Para LogReg se considera eliminar una.'
)
panel_relevancia = panel('relevancia',
    'Ranking combina punto-biserial + mutual information. '
    'Features en top de ambas métricas = <span class=\"tg\">robustas</span>. '
    'Coincide con AutoML: <b>cred_superados_anio_1er · n_anios_beca · nota_1er_anio</b>.'
)


In [10]:
# ============================================================================
# CELDA 9: GENERAR HTML CON render_base_html
# ============================================================================
# Construye el HTML de M06 con todos los graficos y tablas.
# Navegacion a los demas modulos de la Fase 4.
# ============================================================================

# --- valores para KPIs ---
n_features_str  = str(len(FEATURES_NUM))
top1_pb_name    = df_pb.iloc[0]['feature']
top1_pb_r_str   = '{:.3f}'.format(df_pb.iloc[0]['r_pb'])
top1_mi_name    = df_mi.iloc[0]['feature']
top1_mi_val_str = '{:.3f}'.format(df_mi.iloc[0]['mutual_info'])
n_alta_str      = str(len(pares_altos))
n_vif_grave     = int((df_vif['vif'] > VIF_GRAVE).sum()) if df_vif is not None else 0
n_vif_grave_str = str(n_vif_grave)
kpi_vif_bg      = '#fff5f5' if n_vif_grave > 0 else '#f0fff4'
kpi_vif_border  = '#e53e3e' if n_vif_grave > 0 else '#48bb78'

kpis_html = (
    '<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));'
    'gap:16px;margin-bottom:28px;">'

    '<div style="background:#ebf8ff;border-left:4px solid #3182ce;padding:16px 20px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;letter-spacing:.6px;margin-bottom:4px;">'
    'Features analizadas</div>'
    '<div style="font-size:28px;font-weight:700;color:#1a202c;">' + n_features_str + '</div>'
    '</div>'

    '<div style="background:#ebf8ff;border-left:4px solid #3182ce;padding:16px 20px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;letter-spacing:.6px;margin-bottom:4px;">'
    'Top feature (|r_pb|)</div>'
    '<div style="font-size:18px;font-weight:700;color:#1a202c;">' + top1_pb_name + '</div>'
    '<div style="font-size:12px;color:#4a5568;">r = ' + top1_pb_r_str + '</div>'
    '</div>'

    '<div style="background:#ebf8ff;border-left:4px solid #3182ce;padding:16px 20px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;letter-spacing:.6px;margin-bottom:4px;">'
    'Top feature (MI)</div>'
    '<div style="font-size:18px;font-weight:700;color:#1a202c;">' + top1_mi_name + '</div>'
    '<div style="font-size:12px;color:#4a5568;">MI = ' + top1_mi_val_str + '</div>'
    '</div>'

    '<div style="background:' + kpi_vif_bg + ';border-left:4px solid ' + kpi_vif_border + ';'
    'padding:16px 20px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;letter-spacing:.6px;margin-bottom:4px;">'
    'VIF grave (&gt;10)</div>'
    '<div style="font-size:28px;font-weight:700;color:#1a202c;">' + n_vif_grave_str + '</div>'
    '<div style="font-size:12px;color:#4a5568;">Pares |r|&gt;0.7: ' + n_alta_str + '</div>'
    '</div>'

    '</div>'
)

# --- tabla ranking ---
def _fila_ranking(row):
    sig    = '&#10003;' if row.get('significativa', True) else '&#9888;'
    r_pb_s = '{:.3f}'.format(row['r_pb'])
    mi_s   = '{:.3f}'.format(row['mutual_info'])
    rm_s   = '{:.1f}'.format(row['rango_medio'])
    vif_v  = '{:.1f}'.format(row['vif']) if 'vif' in row and row['vif'] == row['vif'] else '&mdash;'
    cat    = str(row.get('categoria', '&mdash;'))
    c_vif  = '#e53e3e' if 'GRAVE' in cat else '#dd6b20' if 'MODER' in cat else '#2d7d46'
    return (
        '<tr>'
        '<td style="font-weight:600;color:#1a202c;padding:7px 10px;">' + str(row['feature']) + '</td>'
        '<td style="text-align:right;padding:7px 8px;">' + r_pb_s + '</td>'
        '<td style="text-align:center;padding:7px 8px;">' + sig + '</td>'
        '<td style="text-align:right;padding:7px 8px;">' + str(int(row['rango_pb'])) + '</td>'
        '<td style="text-align:right;padding:7px 8px;">' + mi_s + '</td>'
        '<td style="text-align:right;padding:7px 8px;">' + str(int(row['rango_mi'])) + '</td>'
        '<td style="text-align:right;font-weight:600;color:#3182ce;padding:7px 8px;">' + rm_s + '</td>'
        '<td style="text-align:right;padding:7px 8px;">' + vif_v + '</td>'
        '<td style="color:' + c_vif + ';font-size:11px;padding:7px 8px;">' + cat + '</td>'
        '</tr>'
    )

filas_ranking = ''.join(_fila_ranking(row) for _, row in df_export.iterrows())
tabla_ranking_html = (
    '<div style="overflow-x:auto;margin:20px 0;">'
    '<table style="width:100%;border-collapse:collapse;font-size:13px;">'
    '<thead><tr style="background:#edf2f7;color:#4a5568;">'
    '<th style="padding:10px 12px;text-align:left;">Feature</th>'
    '<th style="padding:10px 8px;text-align:right;">r_pb</th>'
    '<th style="padding:10px 8px;text-align:center;">Sig.</th>'
    '<th style="padding:10px 8px;text-align:right;">Rango PB</th>'
    '<th style="padding:10px 8px;text-align:right;">MI</th>'
    '<th style="padding:10px 8px;text-align:right;">Rango MI</th>'
    '<th style="padding:10px 8px;text-align:right;">Rango medio</th>'
    '<th style="padding:10px 8px;text-align:right;">VIF</th>'
    '<th style="padding:10px 8px;text-align:left;">Cat. VIF</th>'
    '</tr></thead>'
    '<tbody>' + filas_ranking + '</tbody>'
    '</table></div>'
)

# --- tabla pares alta correlacion ---
if pares_altos:
    filas_pares = ''
    for f1, f2, r in sorted(pares_altos, key=lambda x: abs(x[2]), reverse=True):
        c_r = '#e53e3e' if abs(r) > 0.9 else '#dd6b20'
        r_s = '{:+.3f}'.format(r)
        filas_pares += (
            '<tr>'
            '<td style="padding:7px 10px;">' + f1 + '</td>'
            '<td style="padding:7px 10px;">' + f2 + '</td>'
            '<td style="color:' + c_r + ';font-weight:600;text-align:right;padding:7px 10px;">' + r_s + '</td>'
            '</tr>'
        )
    tabla_pares_html = (
        '<div style="overflow-x:auto;margin:16px 0;">'
        '<table style="width:100%;border-collapse:collapse;font-size:13px;">'
        '<thead><tr style="background:#edf2f7;">'
        '<th style="padding:8px 12px;text-align:left;">Feature A</th>'
        '<th style="padding:8px 12px;text-align:left;">Feature B</th>'
        '<th style="padding:8px 12px;text-align:right;">r de Pearson</th>'
        '</tr></thead>'
        '<tbody>' + filas_pares + '</tbody>'
        '</table></div>'
    )
else:
    tabla_pares_html = '<p style="color:#2d7d46;font-weight:600;">Sin pares con |r| &gt; 0.70</p>'

# --- bloque VIF ---
if IMG_VIF:
    bloque_vif_html = (
        '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 12px;">'
        'Factor de Inflacion de Varianza (VIF)</h2>'
        '<p style="color:#4a5568;font-size:14px;margin-bottom:16px;">'
        'VIF &gt; 10: multicolinealidad grave. 5-10: moderada. &lt; 5: aceptable.'
        '</p>'
        '<img src="data:image/png;base64,' + IMG_VIF + '"'
        ' style="max-width:700px;width:100%;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.1);" />'
    )
else:
    bloque_vif_html = '<p><em>VIF no disponible (statsmodels no instalado).</em></p>'

# --- Conclusiones dinámicas (se recalculan solas si cambia el dataset) ------

# Top 3 y bottom 3 por rango medio
top3     = df_export.head(3)['feature'].tolist()
top3_str = ', '.join(f'<strong>{f}</strong>' for f in top3)
bot3     = df_export.tail(3)['feature'].tolist()
bot3_str = ', '.join(f'<em>{f}</em>' for f in bot3)

# Pares con alta correlación
if pares_altos:
    pares_desc = '; '.join(
        f'<strong>{f1}</strong> ↔ <strong>{f2}</strong> (r={r:+.3f})'
        for f1, f2, r in sorted(pares_altos, key=lambda x: abs(x[2]), reverse=True)
    )
    conclusion_pares = (
        f'Se detectaron <strong>{len(pares_altos)}</strong> par(es) con '
        f'|r| &gt; {CORR_ALTA:.2f}: {pares_desc}. '
        'Candidatos a revisión en modelos lineales; en Gradient Boosting el impacto es menor.'
    )
else:
    conclusion_pares = (
        f'No se detectaron pares con |r| &gt; {CORR_ALTA:.2f}. '
        'Las features numéricas son suficientemente independientes entre sí.'
    )

# VIF
if df_vif is not None:
    vif_graves_lista = df_vif[df_vif['vif'] > VIF_GRAVE]['feature'].tolist()
    vif_moder_lista  = df_vif[(df_vif['vif'] > VIF_MODERADO) & (df_vif['vif'] <= VIF_GRAVE)]['feature'].tolist()
    conclusion_vif = (
        f'<strong>{len(vif_graves_lista)}</strong> feature(s) con VIF &gt; {VIF_GRAVE} (grave): '
        + (', '.join(f'<em>{f}</em>' for f in vif_graves_lista) if vif_graves_lista else '—')
        + f'. <strong>{len(vif_moder_lista)}</strong> moderada(s): '
        + (', '.join(f'<em>{f}</em>' for f in vif_moder_lista) if vif_moder_lista else '—')
        + f'. Candidatas a eliminación <u>solo en modelos lineales</u>; '
        'en CatBoost/XGBoost/LightGBM se pueden mantener todas.'
    )
else:
    conclusion_vif = 'VIF no calculado (statsmodels no disponible).'

# Independencia general
n_pares_posibles = len(FEATURES_NUM) * (len(FEATURES_NUM) - 1) // 2
pct_indep = (1 - len(pares_altos) / n_pares_posibles) * 100
conclusion_indep = (
    f'Del total de {n_pares_posibles} pares posibles entre las {len(FEATURES_NUM)} '
    f'features numéricas, solo {len(pares_altos)} superan |r| = {CORR_ALTA:.2f} '
    f'({pct_indep:.0f}% son independientes). '
    'Las features del dataset son mayoritariamente no redundantes.'
)

# Dirección del efecto (signo r_pb)
df_pos = df_export[df_export['r_pb'] > 0].sort_values('r_pb', ascending=False)
if not df_pos.empty:
    top_riesgo   = df_pos.iloc[0]['feature']
    top_riesgo_r = df_pos.iloc[0]['r_pb']
    conclusion_signo = (
        f'La feature <strong>{top_riesgo}</strong> (r_pb = {top_riesgo_r:+.3f}) '
        'tiene signo positivo: valores más altos se asocian a <em>más</em> abandono (factor de riesgo). '
        'El resto de las features relevantes son negativas: más valor = menos abandono (factor protector).'
    )
else:
    conclusion_signo = 'Todas las features relevantes son protectoras (r_pb negativo).'

bloque_conclusiones = (
    '<div style="background:#fffbeb;border:1px solid #f6e05e;border-radius:8px;'
    'padding:20px;margin-top:32px;">'
    '<h3 style="color:#744210;font-size:15px;margin:0 0 14px;">📋 Conclusiones — '
    f'Módulo 6 ({len(FEATURES_NUM)} features, dataset_final_tfm.parquet)</h3>'

    '<p style="color:#4a5568;font-size:14px;margin:0 0 10px;">'
    f'<strong>🏆 Top 3 más relevantes (rango medio PB+MI):</strong> {top3_str}. '
    'Son las que más información aportan sobre el abandono según ambas métricas.</p>'

    '<p style="color:#4a5568;font-size:14px;margin:0 0 10px;">'
    f'<strong>⬇️ Menos relevantes:</strong> {bot3_str}. '
    'Poca señal estadística; se revisará su inclusión en Fase 5.</p>'

    '<p style="color:#4a5568;font-size:14px;margin:0 0 10px;">'
    f'<strong>⚠️ Pares con alta correlación:</strong> {conclusion_pares}</p>'

    '<p style="color:#4a5568;font-size:14px;margin:0 0 10px;">'
    f'<strong>📐 Multicolinealidad (VIF):</strong> {conclusion_vif}</p>'

    '<p style="color:#4a5568;font-size:14px;margin:0 0 10px;">'
    f'<strong>✅ Independencia general:</strong> {conclusion_indep}</p>'

    '<p style="color:#4a5568;font-size:14px;margin:0;">'
    f'<strong>➕/➖ Dirección del efecto:</strong> {conclusion_signo}</p>'
    '</div>'
)

# --- Cuerpo completo ---
cuerpo = (
    js_css_html
    +
    kpis_html

    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:28px 0 12px;">'
    'Heatmap de correlacion de Pearson</h2>'
    '<p style="color:#4a5568;font-size:14px;margin-bottom:16px;">'
    'Triangulo inferior de la matriz de correlaciones. '
    'Azul = positiva, rojo = negativa. '
    'Detecta redundancias entre features antes del modelado.</p>'
    '<img src="data:image/png;base64,' + IMG_HEATMAP + '"'
    ' style="max-width:750px;width:100%;display:block;margin:0 auto;'
    'border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.1);" />'

    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 12px;">'
    'Clustermap jerarquico</h2>'
    '<p style="color:#4a5568;font-size:14px;margin-bottom:16px;">'
    'Agrupa las features por similitud de correlacion. '
    'Grupos con alta correlacion interna son candidatos a reduccion por redundancia.</p>'
    '<img src="data:image/png;base64,' + IMG_CLUSTERMAP + '"'
    ' style="max-width:750px;width:100%;display:block;margin:0 auto;'
    'border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.1);" />'

    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 12px;">'
    'Relevancia de features respecto al target</h2>'
    '<p style="color:#4a5568;font-size:14px;margin-bottom:16px;">'
    '<strong>Izquierda:</strong> Correlacion point-biserial — relacion lineal con el target. '
    '<strong>Derecha:</strong> Informacion mutua — captura dependencias no lineales.</p>'
    '<img src="data:image/png;base64,' + IMG_RELEVANCIA + '"'
    ' style="max-width:850px;width:100%;display:block;margin:0 auto;'
    'border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.1);" />'

    + bloque_vif_html

    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 12px;">'
    'Tabla de ranking unificado</h2>'
    '<p style="color:#4a5568;font-size:14px;margin-bottom:8px;">'
    'Ranking combinado por point-biserial e informacion mutua. '
    'El <strong>rango medio</strong> identifica las features mas robustas para la Fase 5.</p>'
    + tabla_ranking_html

    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 12px;">'
    f'Pares con alta correlacion (|r| &gt; {CORR_ALTA:.2f})</h2>'
    '<p style="color:#4a5568;font-size:14px;margin-bottom:8px;">'
    'Variables muy correlacionadas pueden generar multicolinealidad en modelos lineales. '
    'En gradient boosting el impacto es menor, pero se documentan para el tribunal.</p>'
    + tabla_pares_html

    + bloque_conclusiones
    + h2btn("Factor de Inflacion de Varianza (VIF)", "vif")
    + panel_vif
    + h2btn("Heatmap de correlacion de Pearson", "heatmap")
    + panel_heatmap
    + h2btn("Relevancia de features vs Target", "relevancia")
    + panel_relevancia
)

from src.html import generar_html_navegacion_completa, guardar_html

nav_fases, nav_modulos = generar_html_navegacion_completa(fase_activa='fase4', modulo_activo='m06')

html_final = render_base_html(
    titulo='F4·M06 — Correlaciones y relevancia de features',
    icono='🔗',
    subtitulo='Fase 4 · EDA Final · Modulo 6',
    nav_fases=nav_fases,
    nav_modulos=nav_modulos,
    contenido=cuerpo,
    notebook_nombre='f4_m06_correlaciones.ipynb',
    notebook_carpeta='fase4_eda'
)

ruta_html = RUTA_FASE4_HTML / 'm06_correlaciones.html'
guardar_html(html_final, ruta_html)

print('=' * 60)
print('COMPLETADO: F4-M05 Correlaciones')
print('=' * 60)
print(f'  HTML   : {ruta_html}')
print(f'  Parquet: {ruta_parquet}')
print(f'  Features analizadas: {n_features_str}')
print(f'  Pares |r| > CORR_ALTA0   : {n_alta_str}')
print(f'  VIF grave (>10)    : {n_vif_grave_str}')
print()
print('Siguiente: f4_m07_seleccion.ipynb')


✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m06_correlaciones.html
COMPLETADO: F4-M05 Correlaciones
  HTML   : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m06_correlaciones.html
  Parquet: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\f4_m06_correlaciones_tabla.parquet
  Features analizadas: 16
  Pares |r| > CORR_ALTA0   : 3
  VIF grave (>10)    : 6

Siguiente: f4_m07_seleccion.ipynb
